[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/notebooks/quickstart.ipynb)

# FRIDA-Decisions quickstart

Structured decisions over text — **choice**, **score**, **yes/no** and **ranking** — from one
encoder pass. This notebook installs the package, loads the model from the Hugging Face Hub,
runs a first request, shows each question type, scores a 243-intent catalog, and runs the
int8 ONNX model on CPU.

On Colab, a GPU runtime (Runtime → Change runtime type → T4 GPU) makes the torch cells fast;
the ONNX cell runs on CPU either way.

In [ ]:
!pip install -q "frida-decisions[torch,onnx] @ git+https://github.com/ai-forever/FRIDA-Decisions"

In [ ]:
import json, os, time
import torch
import frida_decisions
from frida_decisions import Judge, OnnxJudge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")

## First request

A request is a `state` (the text being judged) and named `questions`. All questions are
answered from the same encoder pass.

In [ ]:
request = {
    "state": "Добрый день. Третий день не могу войти в личный кабинет: пишет, что пароль неверный, "
             "а письмо для сброса не приходит. Из-за этого не могу скачать счёт, срок оплаты завтра.",
    "questions": {
        "team": {"type": "choice", "instructions": "В какую команду направить обращение?",
                 "criteria": {"auth": "доступ к аккаунту, вход, пароли",
                              "billing": "счета, оплата, возвраты",
                              "shipping": "доставка заказов"}},
        "angry": {"type": "noul", "instructions": "Автор раздражён?"},
    },
}
response = judge(request)
print(json.dumps(response["answers"], ensure_ascii=False, indent=1))
print(response["usage"])

## The four question types

| type | criteria | answer |
|---|---|---|
| `choice` | `{id: description}` | `choice` (id), `probabilities`, `confidence` |
| `score` | `[level 0, level 1, ...]` | `score` (expected level), `probabilities`, `legend` |
| `noul` (yes/no) | optional `{"true": ..., "false": ...}` | `noul` = probability of yes |
| `ranking` | `{id: text}` or `[text, ...]` | `ranking` (ids, best first), `scores`, `probabilities` |

In [ ]:
EXAMPLES = {
    "choice": {
        "state": "Здравствуйте! Вчера оплатил заказ №5512 картой, деньги списались дважды. Верните, пожалуйста, лишнее списание.",
        "questions": {
            "topic": {
                "type": "choice",
                "instructions": "Какая тема обращения?",
                "criteria": {
                    "billing": "оплата, списания, возвраты денег",
                    "delivery": "доставка и сроки получения заказа",
                    "account": "вход в аккаунт, пароль, личные данные",
                    "product": "качество и характеристики товара"
                }
            }
        }
    },
    "score": {
        "state": "Сервер с базой заказов не отвечает уже сорок минут, клиенты не могут оформить покупку, каждая минута — потерянные продажи.",
        "questions": {
            "urgency": {
                "type": "score",
                "instructions": "Оцени срочность обращения по шкале ниже.",
                "criteria": [
                    "Очень низкая: работа не страдает, ответить можно когда угодно.",
                    "Низкая: неудобство небольшое, ответить можно на этой неделе.",
                    "Средняя: работа затруднена, но обходной путь есть.",
                    "Высокая: работа заблокирована, обходного пути нет.",
                    "Критическая: потери растут с каждым часом."
                ]
            }
        }
    },
    "noul": {
        "state": "Курьер опоздал на два часа и даже не позвонил. Больше заказывать у вас не буду.",
        "questions": {
            "complaint": {
                "type": "noul",
                "instructions": "Это жалоба?"
            }
        }
    },
    "ranking": {
        "state": "Как удалить накипь из чайника?",
        "questions": {
            "best": {
                "type": "ranking",
                "instructions": "Какой фрагмент лучше всего отвечает на вопрос?",
                "criteria": {
                    "p1": "Налейте в чайник воду с двумя ложками лимонной кислоты, вскипятите и оставьте на час, затем промойте.",
                    "p2": "Электрический чайник нельзя погружать в воду целиком: это опасно для нагревательного элемента.",
                    "p3": "Чай лучше заваривать водой температурой 85–95 градусов, а не крутым кипятком.",
                    "p4": "Уксус тоже растворяет накипь: смешайте его с водой один к одному, прокипятите и тщательно сполосните."
                }
            }
        }
    }
}

for kind, example in EXAMPLES.items():
    (qid, answer), = judge(example)["answers"].items()
    print(kind, "->", {k: v for k, v in answer.items() if k not in ("probabilities", "legend")})

## Filtering, not just ordering

`probabilities` split one unit among the candidates, so they cannot filter: with 250 candidates a confident winner still reads 0.09. `scores` can — each is the raw margin of one candidate, computed independently of the rest of the list.

Their *level*, though, is not calibrated. `ranking` is trained with a listwise softmax, which depends only on differences inside one list, so nothing pins where a request's margins sit; on razvilka that level runs from −1.88 to +10.84. Subtracting the level of the request's own candidates fixes most of it, because in a retriever's top-k most candidates are irrelevant: pooled AUC goes from 0.923 to 0.972, and the best single threshold from F1 0.769 to 0.882 (0.855 held out). An anchor candidate with a fixed text does not work — see [`tools/ranking_threshold.py`](https://github.com/ai-forever/FRIDA-Decisions/blob/main/tools/ranking_threshold.py).

The threshold belongs to the shape of the list, not to the model: the cell below keeps both passages that answer the question out of nine, and then shows the same threshold keeping nobody when only the four original passages are in the list.

In [ ]:
QUESTION = {"type": "ranking",
            "instructions": "Какой фрагмент лучше всего отвечает на вопрос?"}
FOUR = EXAMPLES["ranking"]["questions"]["best"]["criteria"]

# A retriever's top-k for the same question: the four passages above plus five
# that do not answer it, so the relevant ones are a minority.
RETRIEVED = dict(
    FOUR,
    p5="Накипь образуется из солей кальция и магния, растворённых в воде; в жёсткой воде её больше.",
    p6="Гарантия на бытовую технику не покрывает поломки из-за нарушения правил эксплуатации.",
    p7="Для заварки зелёного чая возьмите воду около 80 градусов и настаивайте две минуты.",
    p8="Фильтр-кувшин смягчает воду и поэтому реже приходится чистить бытовые приборы.",
    p9="Доставка заказов по городу занимает от одного до трёх рабочих дней.",
)

# Fitted on razvilka, on lists where one candidate in five is relevant.
THRESHOLD = 3.5


def above_the_list(criteria):
    """(id, margin, distance above the list) for one ranking request, best first."""
    request = {"state": "Как удалить накипь из чайника?",
               "questions": {"best": dict(QUESTION, criteria=criteria)}}
    scores = judge(request)["answers"]["best"]["scores"]
    level = sum(scores.values()) / len(scores)
    return sorted(((k, v, v - level) for k, v in scores.items()), key=lambda r: -r[1])


for key, margin, distance in above_the_list(RETRIEVED):
    print(f"{key}  margin {margin:+6.2f}  above the list {distance:+6.2f}  "
          f"{'kept' if distance >= THRESHOLD else ''}")

# The same four passages alone: two of the four answer the question, so they pull
# the level up and nothing clears the threshold. Note the margins themselves are
# unchanged -- a candidate does not know what else is in the list.
best = above_the_list(FOUR)[0]
print(f"\nthe four passages alone: best is {best[0]} at {best[2]:+.2f} above the list, "
      f"so nothing clears {THRESHOLD}")

## A big catalog: 243 intents in one request

Packing puts all options of a question into a few rows next to one copy of the state, and the
state cache encodes the state once. Options never attend to each other, so the result is the
same as scoring every option on its own — only cheaper.

In [ ]:
import urllib.request
from pathlib import Path

local = [p for p in (Path("examples/data/intent_catalog.json"),
                     Path("../examples/data/intent_catalog.json")) if p.exists()]
if local:                       # running from a clone of the repository
    catalog = json.loads(local[0].read_text(encoding="utf-8"))
else:
    url = "https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/main/examples/data/intent_catalog.json"
    catalog = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))

counts = judge.token_counts(catalog["request"])
print(f"{counts['candidates']} intents; encoder tokens: one sequence per intent = {counts['naive']}, "
      f"packed = {counts['packed']}, packed + state cache = {counts['cached']}")

for attempt in ("first call", "same state again (cache hit)"):
    started = time.perf_counter()
    response = judge(catalog["request"])
    print(f"{attempt}: {time.perf_counter() - started:.2f}s", response["usage"]["state_cache"])

answer = response["answers"]["intent"]
top = sorted(answer["probabilities"].items(), key=lambda kv: -kv[1])[:5]
print("choice:", answer["choice"], "| expected:", catalog["expected"]["intent"])
print("top-5:", [(k, round(p, 3)) for k, p in top])

## CPU: int8 ONNX

`OnnxJudge` has the same API and needs only `onnxruntime` at inference. The graph is
quantised to int8 with per-token activation scales.

In [ ]:
onnx_judge = OnnxJudge.from_pretrained(MODEL, threads=os.cpu_count())
started = time.perf_counter()
onnx_answers = onnx_judge(request)["answers"]
print(f"{time.perf_counter() - started:.2f}s")
print(json.dumps(onnx_answers, ensure_ascii=False, indent=1))